reticulate::use_python("/opt/homebrew/Caskroom/miniforge/base/envs/python_scanpy/bin/python")

## Load libraries

In [ ]:
library("data.table")
lapply(c("dplyr","Seurat","HGNChelper","openxlsx"), library, character.only = T)

In [ ]:
out_dir <- "~/Analysis/CellAnnoUsingScType_AfterExclude"

system(paste0("mkdir -p ", out_dir))

## Load R sources from github repository of ScType 

In [ ]:
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/gene_sets_prepare.R"); source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/sctype_score_.R")

In [ ]:
base_dir <- "~/CODEX_08262025/Analysis/Pre_asinh_AfterExclude/"

## Load Expression data

In [ ]:
expr_data <- fread(paste0(base_dir, "expr_data.csv"), 
                   header = T,
                   sep = ",") %>% as.data.frame()
rownames(expr_data) <- expr_data$V1
expr_data <- expr_data[, -1] %>% as.matrix
expr_data %>% head

In [ ]:
rownames(expr_data)

## Load positive and negative gene set markers

In [ ]:
gs_list <- openxlsx::read.xlsx("~/CODEX_08262025/ScTypeDB_cleaned_DFA_updated_09152025.xlsx", 
                               sheet = "Sheet3")
gs_list <- gs_list[gs_list$tissueType == "DFA", ]
nrow(gs_list)

In [ ]:
gs_positive <- lapply(gs_list$cellName, 
                              function(x) {
                                  genes <- gs_list[gs_list$cellName == x, ]$geneSymbolmore1
                                  stringr::str_split(genes, ",", simplify = T)[1, ]
                              })
names(gs_positive) <- gs_list$cellName
gs_positive

In [ ]:
gs_neg <- lapply(gs_list$cellName, 
                              function(x) {
                                  genes <- gs_list[gs_list$cellName == x, ]$geneSymbolmore2
                                  if(all(is.na(genes))) {
                                      NULL
                                  } else {
                                      stringr::str_split(genes, ",", simplify = T)[1, ]
                                  }  
                              })
names(gs_neg) <- gs_list$cellName
gs_neg

In [ ]:
names(gs_neg)[duplicated(names(gs_neg))]

## Compute the cell type scores based on markers

In [ ]:
es.max <- sctype_score(scRNAseqData = expr_data, 
                       scaled = F, gs = gs_positive, gs2 = gs_neg)


In [ ]:
View(es.max)

## Load cell meta data

In [ ]:
metadata <- fread(paste0(base_dir, "cellmetadata.csv"), 
                   header = T,
                   sep = ",") %>% as.data.frame()
rownames(metadata) <- metadata$V1
metadata <- metadata[, -1]
metadata %>% head 

## Aggregate cell type scores from cell levels to cluster levels

In [ ]:
# merge by cluster
cL_resutls <- do.call("rbind", lapply(unique(metadata$leiden_1), function(cl){
    es.max.cl = sort(rowSums(es.max[ ,rownames(metadata[metadata$leiden_1 == cl, ])]), decreasing = !0)
    head(data.frame(cluster = cl, type = names(es.max.cl), scores = es.max.cl, ncells = sum(metadata$leiden_1==cl)), 15)
}))

## Extract top scores of each cluster

In [ ]:
sctype_scores <- cL_resutls %>% group_by(cluster) %>% top_n(n = 1, wt = scores) 
sctype_scores$type[as.numeric(as.character(sctype_scores$scores)) < sctype_scores$ncells/5] <- "Unknown"

In [ ]:
sctype_scores$scoresByCell <- sctype_scores$scores/sctype_scores$ncells

In [ ]:
sctype_scores[sctype_scores$cluster %in% sctype_scores$cluster[duplicated(sctype_scores$cluster)],]

In [ ]:
sctype_scores[sctype_scores$cluster == "32", ]
sctype_scores %>% arrange(desc(scoresByCell))

table(sctype_scores$type)

## Assign cell labels

In [ ]:
metadata$celltypes <- "Unknown"

for(cl in sctype_scores$cluster) {
    celltype <- sctype_scores$type[sctype_scores$cluster == cl]
    metadata$celltypes[metadata$leiden_1 %in% cl] <- celltype
}

metadata$celltypes[metadata$leiden_1 == "21"] <- "Mast"

## Write out cell meta data with cell lables to the file

In [ ]:
fwrite(x = data.frame(cell_idx = rownames(metadata), metadata, check.names = F), 
       file = paste0(out_dir, "/cell_metadata_wCelltype_basedonArcSin.tsv"), 
       sep = "\t", quote = F, row.names = F, col.names = T)

In [ ]:
table(metadata$celltypes, metadata$condition)